---
title: Image Captioning with Frozen Models
description: |
  Week 4 of the residency. Take a vision model and a language model, freeze both, and train only the small adapter that connects them.
author: Rosh Beed
date: '2026-06-29'
image: figures/slide-03.png
categories:
  - multimodal
  - vision
  - language
  - week-4
jupyter: python3
---


Week 4's title slide reads *multimodal transfer learning*, and its subtitle is the
part that matters: **merge models for "novel" tasks.**

Not train a model for a new task. Merge existing models, so that between them they
do something neither was trained for.

[Week 3](../2026-06-22-vision-transformers/) built a vision transformer from
scratch and trained it end to end.

![](figures/slide-02.png){fig-alt="The two week 3 tasks: an encoder mapping a digit image to the label 4, and an encoder-decoder mapping a three-digit image to a sequence."}

That works when the task is small and cheap to supervise. Here is the new one.

![](figures/slide-03.png){fig-alt="A photograph of a little girl climbing into a wooden playhouse, feeding a box labelled \"Magic\", with the output \"A little girl climbing into a wooden playhouse.\""}

There's no dataset big enough to teach vision and language from scratch on a
residency budget. There's also no need. Models that understand images already
exist. So do models that write English.

The task is to connect them.

## Where to Join Them

![](figures/slide-04.png){fig-alt="Two architectures side by side. On the left, a cat image through an Encoder, with the Decoder containing a block labelled xAtt, producing CAT. On the right, the same but with the Decoder containing sAtt."}

This is the decision the week turns on, and both options are already familiar.

**xAtt, cross-attention.** A dedicated attention layer inside the decoder that
looks at the encoder's output at every generation step. This is exactly what [the
multi-digit reader](../2026-06-25-encoder-decoder-transformers/) did last week.

**sAtt, self-attention.** Project the image into the language model's embedding
space and put it in the sequence as if it were a token. Ordinary self-attention
then does the rest, and the language model is not modified at all.

The second is cheaper in every way that matters here.

* No new attention weights to initialise
* No architectural change to a model you did not train
* The only new parameters sit between the two models' widths

LLaVA and PaLiGemma both take this path. So did I.

## What CLIP Already Knows

![](figures/slide-06.png){fig-alt="CLIP's Figure 1: contrastive pre-training on image and caption pairs, then building a classifier from label text, then zero-shot prediction on a new image."}

CLIP is trained by pushing an image and its caption to the same place in a shared
space, over 400 million pairs. That means its image vectors are already organised
by *what the picture is about*, in a space that was built alongside text.

So the adapter is not being asked to teach a language model to see. It is being
asked to translate between two coordinate systems that were each built separately
and happen to describe overlapping things.

The week's actual assignment was:

* Learn how to use the hidden state from ViT or CLIP
* Code the decoder from scratch
* Create synthetic datasets and save them on Hugging Face
* Train and experiment with datasets and alignment
* Use a pretrained Qwen model as the decoder

## Building One

The real service uses CLIP and Qwen3-0.6B with a 1.58M-parameter adapter between
them. Neither fits in a page build.

So the rest of this post builds the same arrangement out of two tiny models. Both
are trained here, on separate tasks, so they genuinely have never met.

The vision side learns to classify digits. The language side is a character-level
model trained on one sentence pattern, `the digit is <word>`, and nothing else. It
knows the template and the ten words, and it has never seen an image.

First, the data and the sentence the language model will learn.

In [ ]:
#| warning: false

# Setup, inlined rather than imported so this notebook runs on its own.
# Keep it folded; nothing below it depends on anything outside this file.

import torch.nn as nn
import visualtorch
import matplotlib.pyplot as plt

# --- chart styling -------------------------------------------------------
# Categorical slots of a CVD-validated palette: blue, orange, aqua, purple.
COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#8a63d2"]
MUTED, GRID, AXIS = "#5b6570", "#e6e6e3", "#d5d5d1"


def style_axes(ax, xlabel=None, ylabel=None, grid="y"):
    """Strip an axes back to the ink that carries information."""
    if xlabel:
        ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    if grid:
        ax.grid(axis=grid, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    return ax


def figure(width=7.0, height=4.2, **kw):
    fig, ax = plt.subplots(figsize=(width, height), **kw)
    return fig, ax

# --- visualtorch, configured ---------------------------------------------
# Dropout is hidden: nn.TransformerEncoderLayer exposes its internal Dropout as a
# traceable leaf, every model here builds it with dropout=0.0, and drawing a no-op
# layer says it is part of the architecture when it is not. Uncoloured it also took
# visualtorch's default orange, near enough to MultiheadAttention's to be confusing.
COLOUR_MAP = {
    nn.Linear: {"fill": COLOURS[0]},
    nn.MultiheadAttention: {"fill": COLOURS[1]},
    nn.Embedding: {"fill": COLOURS[3]},
    nn.LayerNorm: {"fill": "#aeb6bf"},
    nn.GELU: {"fill": COLOURS[2]},
    nn.ReLU: {"fill": COLOURS[2]},
    nn.Flatten: {"fill": "#aeb6bf"},
    nn.Dropout: {"fill": "#e6e6e3"},
}
_COMMON = dict(color_map=COLOUR_MAP, connector_fill="#c3c9d0", background_fill="white",
               font_color=MUTED, legend=True, show_dimension=True,
               type_ignore=[nn.Dropout])


def diagram(model, input_shape, style="graph", **overrides):
    """Render `model` as a PIL image in this site's colours.

    `graph` draws real neurons and suits a short stack; `flow` draws volumetric
    blocks whose size tracks the layer's and stays readable on a deep one. `flow`
    reports a 3-D activation as (1, 1, width), losing the sequence length, so its
    shape labels are off. `level_gap=1` keeps a residual connection drawn close to
    the blocks it skips.
    """
    if style == "graph":
        settings = dict(node_size=24, layer_spacing=110, node_spacing=8,
                        ellipsize_after=5, **_COMMON)
    else:
        settings = dict(spacing=26, scale_xy=2.4, max_xy=280, one_dim_orientation="y",
                        level_gap=1, **_COMMON)
        settings["show_dimension"] = False
    settings.update(overrides)
    return visualtorch.render(model, input_shape=input_shape, style=style, **settings)

# --- block diagrams ------------------------------------------------------
FROZEN, EDGE = "#c3ccd6", "#aeb6bf"


def canvas(width=8.4, height=3.4, panels=1, titles=()):
    """One axes per panel, each an empty 0-to-1 grid."""
    fig, axes = plt.subplots(1, panels, figsize=(width, height), squeeze=False)
    axes = list(axes[0])
    for ax, title in zip(axes, list(titles) + [None] * panels):
        if title:
            ax.set_title(title, fontsize=10, color=MUTED, pad=10)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")
    return (fig, axes[0]) if panels == 1 else (fig, axes)


def block(ax, x, y, w, h, label, colour=None, note=None, text="white", size=9):
    """A labelled rectangle. `note` is a smaller line underneath it."""
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=colour or COLOURS[0],
                               edgecolor="none"))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", color=text,
            fontsize=size, linespacing=1.4)
    if note:
        ax.text(x + w / 2, y - 0.045, note, ha="center", va="top", color=MUTED,
                fontsize=8)


def arrow(ax, start, end, label=None, size=8):
    """A connector between two points, each an (x, y) pair."""
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="-|>", color=EDGE, linewidth=1.3,
                                shrinkA=2, shrinkB=2))
    if label:
        ax.text((start[0] + end[0]) / 2, (start[1] + end[1]) / 2 + 0.05, label,
                ha="center", va="bottom", color=MUTED, fontsize=size)


def caption(ax, text, y=0.04, size=9):
    """A line under a panel, for what the panel is arguing."""
    ax.text(0.5, y, text, ha="center", va="center", color=MUTED, fontsize=size)


import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download

# Every build re-executes this page, so a result that shifts between runs would let
# the prose and the output disagree. Torch's multithreaded CPU reductions add floats
# in whatever order the threads finish in; over a training loop that compounds into a
# different model. One thread makes the run reproducible.
torch.set_num_threads(1)

REVISION = "f705fed08827ff6c36e3b5329495c943a5e544e8"
data = np.load(hf_hub_download("roshbeed/ai-residency-blog-data", "mnist/mnist-small.npz",
                               repo_type="dataset", revision=REVISION))

x_train = torch.from_numpy(data["x_train"]).float() / 255.0
y_train = torch.from_numpy(data["y_train"]).long()
x_test = torch.from_numpy(data["x_test"]).float() / 255.0
y_test = torch.from_numpy(data["y_test"]).long()

WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
characters = sorted(set("".join(WORDS) + " the digit is "))
VOCAB = ["<bos>", "<eos>"] + characters
INDEX = {c: i for i, c in enumerate(VOCAB)}

def encode(sentence):
    return [INDEX["<bos>"]] + [INDEX[c] for c in sentence] + [INDEX["<eos>"]]

captions = [encode(f"the digit is {w}") for w in WORDS]
LENGTH = max(len(c) for c in captions)
CAPTIONS = torch.tensor([c + [INDEX["<eos>"]] * (LENGTH - len(c)) for c in captions])

print(f"{len(VOCAB)} vocabulary entries, captions {LENGTH} tokens long")
print(f"example: {''.join(VOCAB[i] for i in CAPTIONS[7] if VOCAB[i] not in ('<bos>', '<eos>'))}")


Now the vision model. It's a two-layer network trained to classify digits. The
number to watch is its accuracy, because it's the ceiling for everything that
follows.

In [ ]:
DIM, PREFIX = 64, 8

class VisionModel(nn.Module):
    """Trained to classify digits. Its hidden layer is the image embedding."""

    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                                   nn.Linear(128, DIM), nn.ReLU())
        self.classifier = nn.Linear(DIM, 10)

    def forward(self, images):
        return self.trunk(images)

torch.manual_seed(0)
vision = VisionModel()
optimiser = torch.optim.Adam(vision.parameters(), lr=1e-3)
generator = torch.Generator().manual_seed(0)

for _ in range(6):
    perm = torch.randperm(len(x_train), generator=generator)
    for i in range(0, len(perm) - 128, 128):
        b = perm[i:i + 128]
        loss = nn.functional.cross_entropy(vision.classifier(vision(x_train[b])), y_train[b])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

with torch.no_grad():
    VISION_CEILING = (vision.classifier(vision(x_test)).argmax(1) == y_test).float().mean().item()
print(f"the vision model classifies digits at {VISION_CEILING:.4f}")

Whatever that number is, it is the one the whole post ends up bumping against.

Then the language model, trained only on those ten sentences. One detail in it
matters later: it's pretrained with the image slots *already present* in the
sequence, filled by a learned placeholder. Without that, inserting an image at
inference would shift every text token one position along, into positional
embeddings the frozen model was never trained with.

In [ ]:
#| warning: false

class LanguageModel(nn.Module):
    """A causal character model. It always has PREFIX slots in front of the text,
    filled by a learned null during pretraining and by the adapter afterwards."""

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(VOCAB), DIM)
        self.positions = nn.Parameter(torch.randn(1, LENGTH + PREFIX, DIM) * 0.02)
        self.null = nn.Parameter(torch.zeros(1, PREFIX, DIM))
        layer = nn.TransformerEncoderLayer(DIM, 4, 4 * DIM, batch_first=True,
                                           norm_first=True, dropout=0.0)
        self.stack = nn.TransformerEncoder(layer, 2)
        self.out = nn.Linear(DIM, len(VOCAB))

    def forward(self, tokens, prefix=None):
        front = self.null.expand(len(tokens), -1, -1) if prefix is None else prefix
        h = torch.cat([front, self.embed(tokens)], dim=1) 
        h = h + self.positions[:, :h.shape[1]]
        mask = nn.Transformer.generate_square_subsequent_mask(h.shape[1])
        return self.out(self.stack(h, mask=mask, is_causal=True))

torch.manual_seed(1)
language = LanguageModel()
optimiser = torch.optim.Adam(language.parameters(), lr=3e-3)

for _ in range(1500):
    batch = CAPTIONS[torch.randint(0, 10, (64,))]
    logits = language(batch[:, :-1])[:, PREFIX:]
    loss = nn.functional.cross_entropy(logits.reshape(-1, len(VOCAB)), batch[:, 1:].reshape(-1))
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

print(f"the language model writes the sentence pattern, final loss {loss.item():.4f}")

Both are now frozen, and a small adapter goes between them. Only the adapter gets
an optimiser. The two models either side never receive a gradient again.

In [ ]:
for p in vision.parameters():
    p.requires_grad_(False)
for p in language.parameters():
    p.requires_grad_(False)

adapter = nn.Sequential(nn.Linear(DIM, DIM), nn.GELU(), nn.Linear(DIM, PREFIX * DIM))

trainable = sum(p.numel() for p in adapter.parameters())
frozen = sum(p.numel() for p in vision.parameters()) + sum(p.numel() for p in language.parameters())
print(f"trainable (adapter): {trainable:,}")
print(f"frozen (both models): {frozen:,}")

So about a sixth of the parameters move, and five sixths of them never will.

That is the whole arrangement:

In [ ]:
#| label: fig-arrangement
#| fig-cap: Two models that have never met, and the only thing between them that learns. The vision model was trained to classify and the language model to write one sentence; neither receives a gradient from here on.
#| fig-alt: Three boxes in a row joined by arrows. The outer two are grey and labelled frozen, for the vision model and the language model. The middle one is blue and labelled trained. An arrow leaves the right-hand box carrying the sentence "the digit is seven".

fig, ax = canvas(width=8.6, height=2.9)

block(ax, 0.02, 0.40, 0.17, 0.30, "vision model\nfrozen", FROZEN,
      note=f"a {DIM}-number summary", text=MUTED)
block(ax, 0.32, 0.40, 0.17, 0.30, "adapter\ntrained", COLOURS[0],
      note=f"{trainable:,} parameters")
block(ax, 0.62, 0.40, 0.17, 0.30, "language model\nfrozen", FROZEN,
      note=f"{PREFIX} slots in front of the text", text=MUTED)

for x in (0.19, 0.49, 0.79):
    arrow(ax, (x, 0.55), (x + 0.11, 0.55))
ax.text(0.915, 0.55, '"the digit\nis seven"', fontsize=9, color=MUTED, va="center")

ax.text(0.105, 0.88, "trained to classify digits", ha="center", fontsize=8.5, color=MUTED)
ax.text(0.405, 0.88, "the only thing that learns", ha="center", fontsize=8.5, color=COLOURS[0])
ax.text(0.705, 0.88, "trained to write one sentence", ha="center", fontsize=8.5, color=MUTED)
ax.text(0.5, 0.08, f"{trainable / (trainable + frozen):.0%} of the parameters "
        f"receive a gradient", ha="center", fontsize=9, color=MUTED)
fig.tight_layout()


Drawn on its own, the trainable part is a two-layer network that ends eight times
wider than it starts, one output vector per slot the language model is holding
open for it.

In [ ]:
#| label: fig-adapter-shape
#| fig-cap: The entire trainable part. It takes the vision model's 64-number summary and produces eight 64-number vectors for the language model to read as if they were text.
#| fig-alt: A row of four blocks. The first three are the same height and the last is visibly taller, the drawing capped well short of the eightfold widening it stands for.

diagram(adapter, input_shape=(1, DIM), style="flow")

Before training the adapter, it's worth seeing what the frozen pair does on its
own. An untrained adapter hands the language model eight vectors of noise, so what
comes back barely depends on the image at all.

In [ ]:
@torch.no_grad()
def caption(n=1000):
    """Generate a caption one character at a time and check the word at the end."""
    prefix = adapter(vision(x_test[:n])).view(n, PREFIX, DIM)
    tokens = torch.full((n, 1), INDEX["<bos>"])
    for _ in range(LENGTH):
        nxt = language(tokens, prefix)[:, -1].argmax(-1, keepdim=True)
        tokens = torch.cat([tokens, nxt], dim=1)

    text = ["".join(VOCAB[i] for i in row[1:LENGTH] if VOCAB[i] not in ("<bos>", "<eos>"))
            for row in tokens]
    correct = sum(t.strip().endswith(WORDS[y]) for t, y in zip(text, y_test[:n].tolist()))
    return correct / n, text

before, examples = caption()
print(f"before training the adapter: {before:.4f}")
print(f"  a sample caption: {examples[0]!r}")
print(f"  distinct captions across all {len(examples):,} images: {len(set(examples))}")

A few distinct captions across a thousand images, and almost none of them name
the right digit. That is the floor the adapter starts from.

Now train only the adapter, on image and caption pairs. The label at the image
position is masked out. The model is never asked to predict the image, only to
predict caption words given it.

In [ ]:
optimiser = torch.optim.Adam(adapter.parameters(), lr=1e-3)
history = []

for epoch in range(24):
    perm = torch.randperm(len(x_train), generator=generator)
    for i in range(0, len(perm) - 128, 128):
        b = perm[i:i + 128]
        prefix = adapter(vision(x_train[b])).view(len(b), PREFIX, DIM)
        target = CAPTIONS[y_train[b]]
        logits = language(target[:, :-1], prefix)[:, PREFIX:]
        loss = nn.functional.cross_entropy(logits.reshape(-1, len(VOCAB)),
                                           target[:, 1:].reshape(-1))
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    history.append(caption()[0])

accuracy, examples = caption()
print(f"after training only the adapter: {accuracy:.4f}\n")
for text, truth in zip(examples[:5], y_test[:5].tolist()):
    word = WORDS[truth]
    article = "an" if word[0] in "aeiou" else "a"
    print(f"  {text!r}   (actually {article} {word})")

From naming no digit at all to naming most of them, and the sentence around the
digit never changed, because it was already right.

Plotting that against the vision model's own accuracy shows where it stops, and
why.

In [ ]:
#| label: fig-adapter
#| fig-cap: The adapter climbing toward the frozen vision model's own accuracy, which it cannot pass.
#| fig-alt: A curve rising steeply from near zero over 24 epochs and flattening out just below an upper dashed line marking what the frozen vision model scores, well above a lower dashed line at 0.1 marking chance.

epochs = range(1, len(history) + 1)
fig, ax = figure(height=4.0)
ax.axhline(VISION_CEILING, color=COLOURS[2], linewidth=1.2, linestyle="--")
ax.axhline(0.1, color=MUTED, linewidth=1.2, linestyle="--")
ax.plot(epochs, history, color=COLOURS[0], linewidth=2)
ax.annotate("what the frozen vision model knows", xy=(1, VISION_CEILING), xytext=(2, 6),
            textcoords="offset points", fontsize=9, color=COLOURS[2])
ax.annotate("guessing one of ten words", xy=(1, 0.1), xytext=(2, 6),
            textcoords="offset points", fontsize=9, color=MUTED)
ax.set_ylim(0, 1.0)
style_axes(ax, "Epoch (adapter only)", "Captions with the right digit")
fig.tight_layout()

## Conclusion

Four things came out of building it.

**The ceiling is the frozen encoder.** The adapter climbs toward the vision model's
own accuracy and flattens out under it. It never sees the image, only what the vision model kept.
Anything the encoder discarded is gone. So picking the encoder matters more than
designing the adapter.

**Fluency comes free, grounding does not.** Every caption after training is correct
English in the right format, and nothing in the adapter's 37,440 parameters learned
to write it — the language model already could. The whole run is spent on *which
word*. That is the only thing it could not already do.

One prefix token was not enough here, which surprised me, because the real project
gets by with a single one. My first version projected the image to one embedding
and it barely beat chance. CLIP's pooled vector is a summary built from 400 million
pairs being read by a model with real depth; mine was a small classifier's hidden
layer being read by two transformer layers. How many tokens you need depends on how
good the summary is and how much model is reading it.

And the positions have to line up, which is the reason for those placeholder slots
in the language model above.

The full project, with CLIP and Qwen3 in place of these two toys, is
[on GitHub](https://github.com/RoshBeed/ai-residency/tree/main/services/multimodal-captioning).
